In [ ]:
import pandas as pd

In [ ]:
# load the data
df = pd.read_csv("attrition_data.csv")

In [ ]:
# shuffle the data
df = df.sample(frac=1).reset_index(drop=True)

In [ ]:
# take a look at columns to:
# 1- find cols to remove: useless (e.g. primary key) or non production cols
# 2- find the target col(s)
# 3- features
df.info()

In [ ]:
# 1- primary key col: 
primary_key_col = "employeenumber"
# it doesn't contribute anything. It will be deleted
df = df.drop(primary_key_col, axis=1)

# 2- target col
target_col = "attrition"

# 3- non production col: none

In [ ]:
%pip install -q scikit-learn

In [ ]:
# train test split
from sklearn.model_selection import train_test_split

X = df.drop(target_col, axis=1)
y = df[target_col]

train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

# dividing into train and val_test together
X_train, X_val_test, y_train, y_val_test = train_test_split(
    X, y, test_size=(val_ratio + test_ratio), random_state=42, stratify=y
)

# diving val_test into val and test separately
X_val, X_test, y_val, y_test = train_test_split(
    X_val_test, y_val_test, test_size=(test_ratio / (val_ratio + test_ratio)), random_state=42, stratify=y_val_test
)

df_train = pd.concat([X_train, y_train], axis=1)
df_val = pd.concat([X_val, y_val], axis=1)
df_test = pd.concat([X_test, y_test], axis=1)

In [ ]:
# some EDA

# looking at number of rows
m, n = X_train.shape
C = y_train.nunique()
print(f"Unique target values (classes):", y_train.unique())

print("number of rows:", m)
print("number of features:", n)
print("number of classes:", C)

# null values
# There are no null values

# identifying numerical and categorical cols
numerical_cols = X_train.select_dtypes(include=["number"]).columns
categorical_cols = X_train.select_dtypes(include=["object"]).columns
print("number of numerical columns:", len(numerical_cols))
print("number of categorical columns:", len(categorical_cols))

In [ ]:
# chi squared test between target and categorical variables
from scipy.stats import chi2_contingency
# H0: independent
# Ha: dependent

alpha = 0.05

dependent_cols = []
independent_cols = []
for categorical_col in categorical_cols:
    contingency_table = pd.crosstab(df[categorical_col], df[target_col])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    if p <= alpha:
        # reject H0
        dependent_cols.append(categorical_col)
    else:
        # accept H0 i.e. independent
        independent_cols.append(categorical_col)

In [ ]:
# 100% stacked bar chart for each category in the col (x-axis) and target categories (y-axis)
import matplotlib.pyplot as plt

# Grid layout
n_cols = 3
n_rows = (len(independent_cols) + n_cols - 1) // n_cols  # ceil division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flatten()

for i, col in enumerate(independent_cols):
    ct = pd.crosstab(df[col], df[target_col], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], legend=(i==0))  # show legend only on first plot
    axes[i].set_title(f'{col} vs {target_col}')
    axes[i].set_ylabel('Proportion')
    axes[i].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))

# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# outliers

rical_cols_with_outliers = []
for numerical_col in numerical_cols:
    summary = df_train[numerical_col].describe()
    minimum = summary["min"]
    twenty_fifth = summary['25%']
    median = summary['50%']
    seventy_fith = summary['75%']
    maximum = summary["max"]
    iqr = seventy_fith - twenty_fifth
    lower_extreme = twenty_fifth - 1.5 * iqr
    upper_extreme = seventy_fith + 1.5 * iqr

    lower_extreme, upper_extreme

    bv = (df[numerical_col] >= upper_extreme) & (df[numerical_col] <= lower_extreme)
    print(f"The column ```{numerical_col}``` has {sum(bv)} outliers")

    if sum(bv) > 0:
        numerical_cols_with_outliers.append({
            "col_name": numerical_col,
            "lower_extreme": lower_extreme,
            "upper_extreme": upper_extreme
        })

In [ ]:
# capping data
# TODO

In [ ]:
# visualization of numerical vs target
# TODO

In [ ]:
# checking for class imbalance
round(y_train.value_counts(normalize=True), 2)

In [ ]:
# addressing class imbalance
# 1. undersampling
from imblearn.under_sampling import RandomUnderSampler

undersampling_ratio = 0.5 # i.e. minority class will be = undersamping_ratio * majority

undersampler = RandomUnderSampler(sampling_strategy=undersampling_ratio, random_state=42)
X_under, y_under = undersampler.fit_resample(X_train, y_train)

print(round(y_under.value_counts(normalize=True), 2))
print(f"There are now {len(X_under)} rows in the dataset. {len(X_train) - len(X_under)} rows have been dropped by undersampling.")

In [ ]:
from imblearn.over_sampling import RandomOverSampler

oversampling_ratio = 1.0 # i.e. the minority class will be = oversampling_ratio * majority class
# not all values work here because small values may only be achievable by undersampling the majority first, which is not what we want here.

oversampler = RandomOverSampler(sampling_strategy=oversampling_ratio, random_state=42)
X_over, y_over = oversampler.fit_resample(X_under, y_under)

print(round(y_over.value_counts(normalize=True), 2))
print(f"There are now {len(X_over)} rows in the the dataset. {len(X_over) - len(X_under)} rows have been added by oversampling.")

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(y_over)

# check if Yes is 1 and No is 0.
print("Class Mapping:", dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))

In [ ]:
y_val = label_encoder.transform(y_val)
y_test = label_encoder.transform(y_test)

In [ ]:
categorical_col_indices = [X_train.columns.get_loc(col) for col in categorical_cols]

In [ ]:
# FP: Model said an employee would be leaving, but the employee is not leaving
# FN: Model said the employee would not be leaving, but the employee left

# Assuming we're creating a reward system, it would be worse to reward an employee we think is staying
# but the employee is actually leaving. so let's try to minimize FN and maximimze recall.

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

In [ ]:
from tqdm.auto import tqdm
from itertools import product

learning_rate = [0.01, 0.033, 0.1]
depth = [2, 4, 6]
l2_leaf_reg = [1, 3, 5]

hps = list(product(learning_rate, depth, l2_leaf_reg))

trained_models = []
metrics = []

max_recall = 0.0
best_model = None

for lr, d, l2 in tqdm(hps, total=len(hps)):
    # define the model
    model = CatBoostClassifier(
        iterations=500,
        learning_rate=lr,
        depth=d,
        l2_leaf_reg=l2,
        cat_features=categorical_col_indices,
        verbose=0
    )
    # train the model
    model.fit(X_over, y_over)
    trained_models.append(model)
    # validation metrics
    y_pred = label_encoder.transform(model.predict(X_val))
    recall = recall_score(y_val, y_pred)
    if recall > max_recall:
        max_recall = recall
        best_model = model
        print(f"new best recall = {max_recall}!")
    metrics.append(recall)

In [ ]:
best_model = trained_models[metrics.index(max(metrics))]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

feature_importance = best_model.get_feature_importance()
feature_names = X_train.columns

# Create a DataFrame
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance})

# Sort by importance
importance_df = importance_df.sort_values(by="Importance", ascending=False)

# Plot using Seaborn
plt.figure(figsize=(10, 5))
sns.barplot(x="Importance", y="Feature", data=importance_df)
plt.title("Feature Importance - CatBoost")
plt.show()

In [ ]:
y_pred = label_encoder.transform(best_model.predict(X_test))

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc_score = roc_auc_score(y_test, y_pred)

print("test data metrics")
print("accuracy:", round(accuracy, 2))
print("precision:", round(precision, 2))
print("recall:", round(recall, 2))
print("f1 score:", round(f1, 2))
print("auc score:", round(auc_score, 2))

In [ ]:
# done!

In [ ]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(best_model, f)

In [ ]:
with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

loaded_model == best_model

In [ ]:
y_pred = label_encoder.transform(loaded_model.predict(X_test))

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc_score = roc_auc_score(y_test, y_pred)

print("test data metrics")
print("accuracy:", round(accuracy, 2))
print("precision:", round(precision, 2))
print("recall:", round(recall, 2))
print("f1 score:", round(f1, 2))
print("auc score:", round(auc_score, 2))

In [ ]:
# single inference:
pred = best_model.predict(X_test.iloc[0])
proba = best_model.predict_proba(X_test.iloc[0])

pred, proba

In [ ]:
# batch inference
preds = best_model.predict(X_test.iloc[:5])
probas = best_model.predict_proba(X_test.iloc[:5])
preds, probas